In [1]:
# Standard libraries
import os
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn utilities
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV,cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif, mutual_info_regression, f_classif, f_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor

# Other utilities
from scipy.stats import randint, uniform
import joblib

# Standard libraries
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd

# Scikit-learn utilities
from sklearn.impute import SimpleImputer

In [2]:
path = "../../data/processed/"
sites = pd.read_parquet(os.path.join(path, "dep_codes.parquet"))
regiones = sites['HERlvl1Code'].drop_duplicates().tolist()
len(regiones)

22

In [3]:
path = "../../notebooks/06_cb_regression/dfs_taxon_p/"
dfs = {}

# read all dataframes and keep them in dfs
for parquet in os.listdir(path):  # List all files in the directory
    if parquet.endswith(".parquet"):
        name = parquet.split(".parquet")[0]  # Get name without extension
        dfs[name] = pd.read_parquet(os.path.join(path, parquet))  # Use os.path.join for paths
        print(f"Loaded {name} with shape {dfs[name].shape}")

from types import SimpleNamespace

# Después de llenar `dfs`:
d = SimpleNamespace(**dfs)

Loaded df_1 with shape (1485, 109)
Loaded df_10 with shape (3926, 148)
Loaded df_11 with shape (1289, 163)
Loaded df_12 with shape (4677, 176)
Loaded df_13 with shape (1003, 172)
Loaded df_14 with shape (7742, 161)
Loaded df_15 with shape (1223, 160)
Loaded df_16 with shape (379, 113)
Loaded df_17 with shape (803, 151)
Loaded df_18 with shape (1164, 152)
Loaded df_19 with shape (500, 134)
Loaded df_2 with shape (352, 97)
Loaded df_20 with shape (508, 209)
Loaded df_21 with shape (2663, 184)
Loaded df_22 with shape (152, 175)
Loaded df_3 with shape (4996, 141)
Loaded df_4 with shape (596, 141)
Loaded df_5 with shape (2313, 131)
Loaded df_6 with shape (2134, 142)
Loaded df_7 with shape (524, 107)
Loaded df_8 with shape (548, 123)
Loaded df_9 with shape (10254, 169)


In [4]:
import numpy as np
import pandas as pd
from typing import Tuple, Dict, Any
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from catboost import CatBoostRegressor

def train_catboost_region_tunueado(
    cleandf: pd.DataFrame,
    target: str = "IBD",
    base = dict,
    test_size: float = 0.20,
    random_state: int = 42,
    cat_params: dict | None = None,
    use_ohe: bool = False,   # False = categóricas nativas (recomendado)
) -> Tuple[CatBoostRegressor, Dict[str, Any], pd.DataFrame]:
    """
    Entrena CatBoostRegressor para una región usando cleandf (con numéricas y categóricas).
    - Separa train (con target) y score (sin target)
    - Preprocesa (imputación cat). Si use_ohe=True, tú haces OHE fuera de esta función.
    - Entrena con early stopping y regularización
    - Devuelve: modelo, métricas en holdout, y scored_df (filas sin target con predicción)
    """

    # -----------------------------
    # 1) separar train / score
    # -----------------------------
    df_train = cleandf[cleandf[target].notna()].copy()
    df_score = cleandf[cleandf[target].isna()].copy()

    # columnas a tirar si existen (evitar leakage)
    drop_cols = [c for c in ["IBD", "IBD_EQR", "IBD_EQR_Status"] if c in cleandf.columns]

    X = df_train.drop(columns=drop_cols + [target], errors="ignore")
    y = df_train[target].astype(float)

    # split
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    # -----------------------------
    # 2) detectar numéricas / categóricas
    # -----------------------------
    num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X_tr.select_dtypes(exclude=[np.number]).columns.tolist()

    # -----------------------------
    # 3) imputación ligera
    # -----------------------------
    # num: mediana no lo ghago por la estructura de taxones ; cat: literal '(missing)' (si usas OHE hazlo igual antes del OHE)
    X_tr_num = X_tr[num_cols].copy()
    X_te_num = X_te[num_cols].copy()


    X_tr_cat = X_tr[cat_cols].copy()
    X_te_cat = X_te[cat_cols].copy()
    X_tr_cat = X_tr_cat.fillna("(missing)")
    X_te_cat = X_te_cat.fillna("(missing)")

    if use_ohe:
        # -------------------------
        # 3.a) OHE (opcional)
        # -------------------------
        # Nota: si quieres OHE aquí, puedes hacerlo con pd.get_dummies para rapidez.
        # (Si ya traes OHE hecho fuera, simplemente deja use_ohe=False)
        X_tr_cat = pd.get_dummies(X_tr_cat, drop_first=False)
        X_te_cat = pd.get_dummies(X_te_cat, drop_first=False)
        # alinear columnas
        X_tr_cat, X_te_cat = X_tr_cat.align(X_te_cat, join="left", axis=1, fill_value=0)
        X_tr_proc = pd.concat([X_tr_num.reset_index(drop=True),
                               X_tr_cat.reset_index(drop=True)], axis=1)
        X_te_proc = pd.concat([X_te_num.reset_index(drop=True),
                               X_te_cat.reset_index(drop=True)], axis=1)
        cat_features = None  # ya no se usan índices de categóricas con OHE
    else:
        # -------------------------
        # 3.b) Categóricas nativas
        # -------------------------
        X_tr_proc = pd.concat([X_tr_num.reset_index(drop=True),
                               X_tr_cat.reset_index(drop=True)], axis=1)
        X_te_proc = pd.concat([X_te_num.reset_index(drop=True),
                               X_te_cat.reset_index(drop=True)], axis=1)
        # índices de columnas categóricas en el DataFrame concatenado
        # (van después de las numéricas)
        cat_features = list(range(len(num_cols), len(num_cols) + len(cat_cols)))

    # -----------------------------
    # 4) CatBoost: base + overrides
    # -----------------------------
    base_params = base 
    if cat_params:
        base_params.update(cat_params)

    model = CatBoostRegressor(**base_params)

    # -----------------------------
    # 5) entrenar con early stopping
    # -----------------------------
    fit_kwargs = dict(
        X=X_tr_proc,
        y=y_tr,
        eval_set=(X_te_proc, y_te),
        use_best_model=True,
    )
    if not use_ohe:
        fit_kwargs["cat_features"] = cat_features  # solo si usamos categóricas nativas

    model.fit(**fit_kwargs)

    # -----------------------------
    # 6) métricas
    # -----------------------------
    pred_tr = model.predict(X_tr_proc)
    pred_te = model.predict(X_te_proc)

    r2_tr = r2_score(y_tr, pred_tr)
    r2_te = r2_score(y_te, pred_te)
    rmse_tr = mean_squared_error(y_tr, pred_tr)
    rmse_te = mean_squared_error(y_te, pred_te)
    mae_tr = mean_absolute_error(y_tr, pred_tr)
    mae_te = mean_absolute_error(y_te, pred_te)

    metrics = {
        "R2_train": float(r2_tr),
        "R2_valid": float(r2_te),
        "MAE_train": float(mae_tr),
        "MAE_valid": float(mae_te),
        "RMSE_train": float(rmse_tr),
        "RMSE_valid": float(rmse_te),
        "best_iterations": int(model.get_best_iteration() or model.tree_count_),
        "gap": float(abs(r2_tr - r2_te)),
        "used_ohe": use_ohe,
    }

    # -----------------------------
    # 7) score para filas sin target
    # -----------------------------
    if not df_score.empty:
        Xs = df_score.drop(columns=drop_cols + [target], errors="ignore")

        Xs_num = Xs[num_cols].copy()
        Xs_cat = Xs[cat_cols].copy().fillna("(missing)")

        if use_ohe:
            Xs_cat = pd.get_dummies(Xs_cat, drop_first=False)
            # alinear con entrenamiento
            Xs_cat = Xs_cat.reindex(columns=X_tr_cat.columns, fill_value=0)

            Xs_proc = pd.concat([Xs_num.reset_index(drop=True),
                                 Xs_cat.reset_index(drop=True)], axis=1)
        else:
            Xs_proc = pd.concat([Xs_num.reset_index(drop=True),
                                 Xs_cat.reset_index(drop=True)], axis=1)

        df_score[target + "_pred"] = model.predict(Xs_proc)
        scored_df = df_score
    else:
        scored_df = pd.DataFrame(columns=list(cleandf.columns) + [target + "_pred"])

    return model, metrics, scored_df

In [5]:
df = d.df_22

target_col = "IBD"  # <-- ajusta
X_all = df.drop(columns=[target_col], errors='ignore')
y_all = df[target_col] if target_col in df.columns else None

# Filas con etiqueta (train) vs sin etiqueta (score)
is_train = y_all.notna() if y_all is not None else pd.Series(False, index=df.index)
X_train, y_train = X_all[is_train], y_all[is_train]
X_score         = X_all[~is_train]

n_train, p = X_train.shape[0], X_train.shape[1]
n_score     = X_score.shape[0]
print(n_train, p, n_score)

120 174 32


In [6]:

basetun = dict(
        loss_function="RMSE",
        eval_metric="RMSE",        # o "R2" si prefieres monitorear R2
        depth=5,                   # 4–6 para 2k x 200
        learning_rate=0.03,        # 0.02–0.05
        iterations=5000,           # early stopping cortará antes
        l2_leaf_reg=10,            # 8–12 reduce gap
        bagging_temperature=1.5,   # diversidad
        subsample=0.8,
        colsample_bylevel=0.7,
        random_strength=1.5,
        max_bin=128,
        random_state=42,
        verbose=0,
    )



In [7]:
#308,96
base2 =  dict(
    loss_function="RMSE",
    eval_metric="RMSE",
    depth=3,                    # 2–4 en 
    learning_rate=0.02,         # suave
    iterations=10000,           # se cortará con ES
    l2_leaf_reg=24,             # 16–28
    min_data_in_leaf=15,        # 12–25
    bootstrap_type="Bayesian",
    bagging_temperature=2.5,    # 2.0–3.0 diversidad por pesos
    colsample_bylevel=0.45,     # 0.35–0.55 (features por árbol)
    random_strength=2.5,        # ruido en splits
    max_bin=64,                 # discretización tosca
    random_state=42,
    verbose=0
)


"""
mode2, metrics2, score2 = train_catboost_region_tunueado(
    d.df_2,
    base = base2,
# False = categóricas nativas (recomendado)
)
metrics2"""

'\nmode2, metrics2, score2 = train_catboost_region_tunueado(\n    d.df_2,\n    base = base2,\n# False = categóricas nativas (recomendado)\n)\nmetrics2'

In [8]:


base22 = alt_bayes_medio = dict(
    loss_function="RMSE", eval_metric="RMSE",
    depth=2, learning_rate=0.018, iterations=12000,
    l2_leaf_reg=24, min_data_in_leaf=18,
    bootstrap_type="Bayesian", bagging_temperature=2.8,
    colsample_bylevel=0.35, random_strength=2.8,
    max_bin=48,
    sampling_frequency="PerTreeLevel",
    leaf_estimation_iterations=2,
    random_state=42, verbose=0
)

"""
mode22, metrics22, score22 = train_catboost_region_tunueado(
    d.df_22,
    base = base22,
# False = categóricas nativas (recomendado)
)
metrics22"""

'\nmode22, metrics22, score22 = train_catboost_region_tunueado(\n    d.df_22,\n    base = base22,\n# False = categóricas nativas (recomendado)\n)\nmetrics22'

In [9]:
import numpy as np
import pandas as pd
from typing import Tuple, Dict, Any
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from catboost import CatBoostRegressor

def train_catboost_region_tunueado_entrenar_evaluar(
    cleandf: pd.DataFrame,
    target: str = "IBD",
    base: Dict[str, Any] = None,
    random_state: int = 42,
    cat_params: Dict[str, Any] | None = None,
    use_ohe: bool = False,
    id_col: str = "SamplingOperations_code",   # <- NUEVO
) -> Tuple[CatBoostRegressor, Dict[str, Any], pd.DataFrame]:

    if base is None:
        base = {}
    base_params = dict(base)
    if cat_params:
        base_params.update(cat_params)

    # --- 1) separar con y sin label
    df_train = cleandf[cleandf[target].notna()].copy()
    df_score = cleandf[cleandf[target].isna()].copy()

    # Asegura que el id exista aunque venga ausente en alguna parte
    if id_col not in cleandf.columns:
        raise KeyError(f"'{id_col}' no está en cleandf.columns")

    # Fuerza a str para que no se corrompa al guardar
    df_train[id_col] = df_train[id_col].astype(str)
    df_score[id_col] = df_score[id_col].astype(str)

    # columnas a tirar para evitar leakage (NO tiramos el id)
    leak_cols = [c for c in ["IBD", "IBD_EQR", "IBD_EQR_Status"] if c in cleandf.columns]

    # --- 2) matriz de entrenamiento (quitamos target, leak y el id)
    cols_to_drop = leak_cols + [target, id_col]
    X_tr = df_train.drop(columns=cols_to_drop, errors="ignore")
    y_tr = df_train[target].astype(float)

    # --- 3) detectar tipos
    num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X_tr.select_dtypes(exclude=[np.number]).columns.tolist()

    # --- 4) imputación ligera
    X_tr_num = X_tr[num_cols].copy()  # NaN OK para CatBoost
    X_tr_cat = X_tr[cat_cols].copy().fillna("(missing)")

    if use_ohe:
        X_tr_cat_ohe = pd.get_dummies(X_tr_cat, drop_first=False)
        X_tr_proc = pd.concat(
            [X_tr_num.reset_index(drop=True), X_tr_cat_ohe.reset_index(drop=True)],
            axis=1
        )
        cat_features = None
        ohe_columns = X_tr_cat_ohe.columns
    else:
        X_tr_proc = pd.concat(
            [X_tr_num.reset_index(drop=True), X_tr_cat.reset_index(drop=True)],
            axis=1
        )
        cat_features = list(range(len(num_cols), len(num_cols) + len(cat_cols))) if cat_cols else None
        ohe_columns = None

    # --- 5) modelo (sin eval_set)
    model = CatBoostRegressor(**base_params)
    fit_kwargs = dict(X=X_tr_proc, y=y_tr, verbose=False)
    if not use_ohe and cat_features:
        fit_kwargs["cat_features"] = cat_features
    model.fit(**fit_kwargs)

    # --- 6) métricas de entrenamiento
    pred_tr = model.predict(X_tr_proc)
    r2_tr = r2_score(y_tr, pred_tr)
    rmse_tr = mean_squared_error(y_tr, pred_tr)
    mae_tr = mean_absolute_error(y_tr, pred_tr)

    try:
        best_iter = model.get_best_iteration()
        if best_iter is None or best_iter < 0:
            best_iter = model.tree_count_
    except Exception:
        best_iter = getattr(model, "tree_count_", None)

    metrics = {
        "R2_train": float(r2_tr),
        "MAE_train": float(mae_tr),
        "RMSE_train": float(rmse_tr),
        "R2_valid": None,
        "MAE_valid": None,
        "RMSE_valid": None,
        "best_iterations": int(best_iter) if best_iter is not None else None,
        "used_ohe": bool(use_ohe),
        "n_train": int(len(y_tr)),
        "n_score": int(len(df_score)),
        "id_col": id_col,
    }

    # --- 7) score para filas sin target (conservando el id)
    if not df_score.empty:
        Xs = df_score.drop(columns=cols_to_drop, errors="ignore")  # <- también quita id_col
        Xs_num = Xs[num_cols].copy()
        Xs_cat = Xs[cat_cols].copy().fillna("(missing)")

        if use_ohe:
            Xs_cat_ohe = pd.get_dummies(Xs_cat, drop_first=False)
            Xs_cat_ohe = Xs_cat_ohe.reindex(columns=ohe_columns, fill_value=0)
            Xs_proc = pd.concat(
                [Xs_num.reset_index(drop=True), Xs_cat_ohe.reset_index(drop=True)],
                axis=1
            )
        else:
            Xs_proc = pd.concat(
                [Xs_num.reset_index(drop=True), Xs_cat.reset_index(drop=True)],
                axis=1
            )

        preds = model.predict(Xs_proc)
        # Devuelve un df compacto con id y predicción (y si quieres, añade más columnas)
        scored_df = df_score[[id_col]].copy()
        scored_df[target + "_pred"] = preds
    else:
        scored_df = pd.DataFrame(columns=[id_col, target + "_pred"])

    return model, metrics, scored_df


In [10]:
predicciones = []
all_metrics = {}  # guardará {region: métricas}

for region in regiones:
    attr = f"df_{int(region)}"           # df_1, df_2, df_18, ...
    if not hasattr(d, attr):
        # fallback por si tus nombres vienen con ceros: df_01
        attr = f"df_{str(region)}"
    if not hasattr(d, attr):
        raise AttributeError(f"No encontré {attr} en d")

    if region == 2:
        base = base2
    elif region == 22:
        base=base22
    else:
        base = basetun

    cleandf = getattr(d, attr)
    model, m, scored = train_catboost_region_tunueado_entrenar_evaluar(cleandf, target='IBD', base = base)  # m = métricas de esa región
    all_metrics[region] = m

    # Mantener índice (SamplingOperations_code) y solo la predicción
    solo_ibd = scored[['SamplingOperations_code','IBD_pred']].copy()
    solo_ibd['region'] = region
    predicciones.append(solo_ibd)
    print(region)

predicciones = pd.concat(predicciones, axis=0)  # índice preservado
predicciones.index.name = 'SamplingOperations_code'

metrics_df = (pd.DataFrame.from_dict(all_metrics, orient='index')
                .reset_index()
                .rename(columns={'index':'region'}))
metrics_df

18
5
4
10
22
9
21
20
12
8
3
17
14
11
13
19
1
7
6
16
15
2


,region,R2_train,MAE_train,RMSE_train,R2_valid,MAE_valid,RMSE_valid,best_iterations,used_ohe,n_train,n_score,id_col
0,18,0.996870,0.091308,0.014023,None,None,None,5000,False,977,187,SamplingOperations_code
1,5,0.996591,0.109927,0.021802,None,None,None,5000,False,2069,244,SamplingOperations_code
2,4,0.997441,0.098158,0.018854,None,None,None,5000,False,494,102,SamplingOperations_code
3,10,0.988737,0.193792,0.092168,None,None,None,5000,False,3389,537,SamplingOperations_code
4,22,0.993394,0.170500,0.050853,None,None,None,12000,False,120,32,SamplingOperations_code
5,9,0.977740,0.178210,0.073057,None,None,None,5000,False,9136,1118,SamplingOperations_code
6,21,0.994894,0.142666,0.035782,None,None,None,5000,False,2332,331,SamplingOperations_code
7,20,0.998395,0.075405,0.010031,None,None,None,5000,False,448,60,SamplingOperations_code
8,12,0.989679,0.170480,0.056281,None,None,None,5000,False,4188,489,SamplingOperations_code
9,8,0.995762,0.095871,0.018064,None,None,None,5000,False,469,79,SamplingOperations_code


In [14]:
metrics_df.to_csv('metrics_train_score.csv', index=False)

In [15]:
predicciones

,SamplingOperations_code,IBD_pred,region
SamplingOperations_code,,,
977,S02000010_20080811,15.062039,18
978,S02000010_20100719,15.438521,18
979,S02000010_20150811,13.819270,18
980,S02000010_20160825,15.086803,18
981,S02000010_20170703,16.036757,18
...,...,...,...
347,S06700075_20120611,20.008280,2
348,S06700094_20210830,18.611227,2
349,S06700590_20210830,18.366359,2


In [16]:
predicciones.to_csv('preds.csv', index = False)

In [17]:
dfull =dict(
    loss_function="RMSE",
    eval_metric="RMSE",
    depth=7,                    # 6–8 suele ir bien
    learning_rate=0.03,
    iterations=8000,            # ES cortará antes
    l2_leaf_reg=10,             # regularización moderada
    min_data_in_leaf=32,        # hojas no tan pequeñas
    bootstrap_type="Bernoulli",
    subsample=0.8,              # row-bagging estable
    colsample_bylevel=0.8,      # feature-bagging moderado
    random_strength=1.2,        # un poco de ruido en splits
    max_bin=128,                # suficiente para 43k
    sampling_frequency="PerTreeLevel",
    random_state=42,
    verbose=0
)

In [19]:
import os

path = "../../notebooks/06_cb_regression/completo/"
dfs = {}

# read 03_CLEAN_COMPLETE_DF.parquet
full = pd.read_parquet(os.path.join(path, "tp_full.parquet"))

In [20]:
model, m, scored = train_catboost_region_tunueado_entrenar_evaluar(full, target='IBD', base = dfull) 

In [22]:
m

{'R2_train': 0.9862324807276721,
 'MAE_train': 0.21458732230260116,
 'RMSE_train': 0.10739976884147795,
 'R2_valid': None,
 'MAE_valid': None,
 'RMSE_valid': None,
 'best_iterations': 8000,
 'used_ohe': False,
 'n_train': 43568,
 'n_score': 5663,
 'id_col': 'SamplingOperations_code'}

In [21]:
scored

,SamplingOperations_code,IBD_pred
43568,S02000010_20080811,13.923305
43569,S02000010_20100719,15.618174
43570,S02000010_20150811,13.511121
43571,S02000010_20160825,14.599235
43572,S02000010_20170703,16.000024
...,...,...
49226,S06940940_20100708,16.145701
49227,S06940940_20230623,15.413047
49228,S06960950_20160629,19.904753
49229,S06960950_20180719,19.983617


In [23]:
scored.to_csv('pred_full_t.csv', index = False)